In [1]:
import numpy as np
import pandas as pd
from scipy.io import loadmat
import astropy.units as u
import re
from scipy.io import savemat
import pickle

In [5]:
h2_lines_dict_total = {'[3,16]': [1418.23, 1513.99, 1593.26, 1621.12],
                       '[4,4]': [1477.05, 1526.55, 1613.72],
                       '[1,7]': [1467.08, 1500.45, 1524.65, 1556.87],
                       '[1,4]': [1431.01, 1446.12, 1489.57, 1504.76],
                       '[0,2]': [1402.65, 1463.83, 1525.15],
                       '[2,12]': [1434.54, 1453.10, 1555.89, 1588.80],
                       '[0,3]': [1395.20, 1407.29, 1468.39],
                       '[4,13]': [1415.33, 1509.45, 1613.99],
                       '[3,0]': [1435.05, 1591.32, 1636.34],
                       '[0,1]': [1398.95, 1460.17, 1521.59]}

dict_line_Bmn = {'1415.33': 0.125,
                 '1509.45': 0.023,
                 '1613.99': 0.092,
                 '1418.23': 0.050,
                 '1513.99': 0.057,
                 '1593.26': 0.122,
                 '1621.12': 0.062,
                 '1477.05': 0.039,
                 '1526.55': 0.033,
                 '1613.72': 0.150,
                 '1467.08': 0.080,
                 '1500.45': 0.101,
                 '1524.65': 0.111,
                 '1556.87': 0.074,
                 '1431.01': 0.058,
                 '1446.12': 0.083,
                 '1489.57': 0.094,
                 '1504.76': 0.115,
                 '1435.05': 0.118,
                 '1591.32': 0.233,
                 '1636.34': 0.099,
                 '1398.95': 0.141,
                 '1460.17': 0.083,
                 '1521.59': 0.032,
                 '1402.65': 0.126,
                 '1463.83': 0.074,
                 '1525.15': 0.029,
                 '1434.54': 0.066,
                 '1453.10': 0.049,
                 '1555.89': 0.077,
                 '1588.80': 0.0119,
                 '1612.39': 0.138,
                 '1617.42': 0.103,
                 '1395.20': 0.096,
                 '1407.29': 0.120,
                 '1468.39': 0.070}

h2_pumping_dict = {'[3,13]': 1213.36* u.AA,'[4,13]':1213.68* u.AA,'[3,16]':1214.47* u.AA,'[4,4]':1214.78* u.AA,'[1,7]':1215.73* u.AA,
                   '[1,4]':1216.07* u.AA,'[3,0]':1217.04* u.AA,'[0,1]':1217.21* u.AA,'[0,2]':1217.64* u.AA,'[2,12]':1217.9* u.AA,'[2,15]':1218.52* u.AA,'[0,3]':1219.09* u.AA}

line_ID_dict = {'1608.44': '(3-9)P(14)',
                '1615.43': '(3-10)R(12)',
                '1415.33': '(4-6)R(12)',
                '1509.45': '(4-8)R(12)',
                '1613.99': '(4-11)R(12)',
                '1418.23': '(3-5)R(15)',
                '1513.99': '(3-7)R(15)',
                '1593.26': '(3-9)R(15)',
                '1621.12': '(3-10)R(15)',
                '1477.05': '(4-8)P(5)',
                '1526.55': '(4-9)P(5)',
                '1613.72': '(4-11)P(5)',
                '1467.08': '(1-6)P(8)',
                '1500.45': '(1-7)R(6)',
                '1524.65': '(1-7)P(8)',
                '1556.87': '(1-8)R(6)',
                '1431.01': '(1-6)R(3)',
                '1446.12': '(1-6)P(5)',
                '1489.57': '(1-7)R(3)',
                '1504.76': '(1-7)P(5)',
                '1435.05': '(3-7)P(1)',
                '1591.32': '(3-10)P(1)',
                '1636.34': '(3-11)P(1)',
                '1398.95': '(0-5)P(2)',
                '1460.17': '(0-6)P(2)',
                '1521.59': '(0-2)P(2)',
                '1402.65': '(0-5)P(2)', 
                '1463.83': '(0-6)P(3)',
                '1525.15': '(0-7)P(3)',
                '1434.54': '(2-5)P(13)',
                '1453.10': '(2-6)R(11)',
                '1555.89': '(2-8)R(11)',
                '1588.80': '(2-8)P(13)',
                '1612.39': '(2-8)P(16)',
                '1617.42': '(2-9)R(14)',
                '1395.20': '(0-5)R(2)',
                '1407.29': '(0-5)P(4)',
                '1468.39': '(0-6)P(4)'}

h2_lines_dict_2011 = {'[3,16]': [1418.23, 1513.99, 1593.26], 
                      '[4,4]': [1477.05, 1613.72], 
                      '[1,7]': [1467.08, 1500.45, 1524.65, 1556.87],
                      '[1,4]': [1431.01, 1446.12, 1489.57],
                      '[0,2]': [1463.83], 
                      '[2,12]': [1453.10, 1555.89, 1588.80],
                      '[0,3]': [1407.29],
                      '[4,13]': [1613.99],
                      '[3,0]': [1591.32, 1636.34],
                      '[0,1]': [1460.17]}

h2_lines_dict_2013 = {'[3,16]': [1418.23, 1513.99, 1593.26],
                      '[4,4]': [1477.05, 1526.55, 1613.72],
                      '[1,7]': [1467.08, 1500.45, 1524.65, 1556.87],
                      '[1,4]': [1431.01, 1446.12, 1489.57, 1504.76],
                      '[0,2]': [1463.83],
                      '[2,12]': [1453.10, 1588.80],
                      '[0,3]': [1407.29],
                      '[4,13]': [1613.99],
                      '[3,0]': [1591.32, 1636.34],
                      '[0,1]': [1398.95, 1460.17, 1521.59]}

h2_lines_dict_2019 = {'[3,16]': [1418.23],
                      '[4,4]': [1477.05],
                      '[1,7]': [1467.08, 1500.45, 1524.65, 1556.87],
                      '[1,4]': [1431.01, 1446.12, 1489.57, 1504.76],
                      '[0,2]': [1463.83],
                      '[0,3]': [1407.29],
                      '[3,0]': [1591.32],
                      '[0,1]': [1398.95, 1460.17, 1521.59]}

h2_lines_dict_2022 = { '[1,7]': [1524.65, 1556.87]}

__Reading in Files__

In [17]:
# file_target_wavelengths = r"C:\Users\zoeyates\H2_models\input_files\wavelengths\wavelengths_2011_1_7.sav"
# data_target_wavelengths = loadmat(file_target_wavelengths)
# print(data_target_wavelengths)

file_target_velocity = r"C:\Users\zoeyates\H2_models\input_files\vel_2011_1_7.pkl"
data_target_velocity = pd.read_pickle(file_target_velocity)
print(data_target_velocity)

# file_target_flux= r"C:\Users\zoeyates\H2_models\input_files\flux_2011_1_7.pkl"
# data_target_flux = pd.read_pickle(file_target_flux)
# print(data_target_flux)

# file_target_flux_err = r"C:\Users\zoeyates\H2_models\input_files\flux_err_2011_1_7.pkl"
# data_target_flux_err = pd.read_pickle(file_target_flux_err)
# print(data_target_flux_err)

data_line_params = pd.read_csv(r"C:\Users\zoeyates\H2_models\input_files\line_parameters\AA_Tau_line_parameters_2011_1_7.csv", sep=",")
print(data_line_params)

{'1467.08': array([-204.11758335, -204.11758335, -204.11758335, ...,  203.64239556,
        203.64239556,  203.64239556]), '1500.45': array([-199.0570808 , -199.0570808 , -199.0570808 , ...,  198.73091234,
        198.73091234,  198.73091234]), '1524.65': array([-196.16071525, -196.16071525, -196.16071525, ...,  196.15425983,
        196.15425983,  196.15425983]), '1556.87': array([-192.29523925, -192.29523925, -192.29523925, ...,  192.67093628,
        192.67093628,  192.67093628])}
     Line_ID ['v', J']  Wavelength   B_mn  Pumping Wavelength  \
0  (1-6)P(8)     [1,7]     1467.08  0.080             1215.73   
1  (1-7)R(6)     [1,7]     1500.45  0.101             1215.73   
2  (1-7)P(8)     [1,7]     1524.65  0.111             1215.73   
3  (1-8)R(6)     [1,7]     1556.87  0.074             1215.73   

   Observed Wavelength  Velocity Shift  
0          1467.198715       24.275723  
1          1500.548933       19.780753  
2          1524.753448       20.355087  
3          1556.97512

__Creating Files__

1) line_params

In [10]:
columns = ['Line_ID', "['v', J']", 'Wavelength', 'B_mn', 'Pumping Wavelength', 'Observed Wavelength', 'Velocity Shift']
def get_info_from_data(file, progression, year, line):
    load_data = np.load(file, allow_pickle=True)
    data = load_data['data'].item()
    line_float = float(line)
    line_str = f"{line_float:.2f}"
    if year in data:
        if line_float in data[year][progression].keys():
            mu = data[year][progression][line_float]['mu']
            vel_shift = ((mu - line_float) / line_float) * 3e5
            B_mn = dict_line_Bmn[line_str]
            line_ID = line_ID_dict[line_str]
            pumping_wavelength = h2_pumping_dict[progression].value
            data_for_csv = pd.DataFrame([{
                columns[0]: line_ID,
                columns[1]: progression,
                columns[2]: line_str,
                columns[3]: B_mn,
                columns[4]: pumping_wavelength,
                columns[5]: mu,
                columns[6]: vel_shift}])
            return data_for_csv

In [11]:

years_list = ['2011', '2013', '2019']
progressions_list = ['[1,7]','[4,13]', '[3,16]', '[4,4]', '[1,4]', '[3,0]', '[0,1]', '[0,2]','[2,12]', '[0,3]']
path_to_files =  r"C:\Users\zoeyates\Downloads\AA_Tau_Results_Files\AA_Tau_Results_Files"
files_dict = {'[1,7]': r"C:\Users\zoeyates\Downloads\AA_Tau_Results_Files\AA_Tau_Results_Files\fitting_parameters_emperical_newstuff1_7.npz",
              '[4,13]': r"C:\Users\zoeyates\Downloads\AA_Tau_Results_Files\AA_Tau_Results_Files\fitting_parameters_emperical_newstuff4_13.npz",
              '[3,16]': r"C:\Users\zoeyates\Downloads\AA_Tau_Results_Files\AA_Tau_Results_Files\fitting_parameters_emperical_newstuff3_16.npz",
              '[4,4]': r"C:\Users\zoeyates\Downloads\AA_Tau_Results_Files\AA_Tau_Results_Files\fitting_parameters_emperical_newstuff4_4.npz",
              '[1,4]': r"C:\Users\zoeyates\Downloads\AA_Tau_Results_Files\AA_Tau_Results_Files\fitting_parameters_emperical_newstuff1_4.npz",
              '[3,0]': r"C:\Users\zoeyates\Downloads\AA_Tau_Results_Files\AA_Tau_Results_Files\fitting_parameters_emperical_newstuff3_0.npz",
              '[0,1]': r"C:\Users\zoeyates\Downloads\AA_Tau_Results_Files\AA_Tau_Results_Files\fitting_parameters_emperical_newstuff0_1.npz",
              '[0,2]': r"C:\Users\zoeyates\Downloads\AA_Tau_Results_Files\AA_Tau_Results_Files\fitting_parameters_emperical_newstuff0_2.npz",
              '[2,12]': r"C:\Users\zoeyates\Downloads\AA_Tau_Results_Files\AA_Tau_Results_Files\fitting_parameters_emperical_newstuff2_12.npz",
              '[0,3]': r"C:\Users\zoeyates\Downloads\AA_Tau_Results_Files\AA_Tau_Results_Files\fitting_parameters_emperical_newstuff0_3.npz"}

for year in years_list: 
    for progressions in progressions_list: 
        df_list = []
        for lines in h2_lines_dict_total[progressions]:
            sub_df = get_info_from_data(files_dict[progressions], progressions, year, lines)
            if sub_df is not None:
                df_list.append(sub_df)
        if len(df_list) !=0:
            df = pd.concat(df_list, ignore_index=True)
            df.to_csv(r'C:\Users\zoeyates\H2_models\input_files\line_parameters\AA_Tau_line_parameters_' + year + '_'+ progressions[1] + '_' + progressions[3] + '.csv', index=False)

2. target_wavelengths

In [14]:
columns_target_wavelengths = ['labwave', 'v_exc', 'j_exc', 'v_ground', 'j_ground']
def create_target_wavelengths():
    for year in years_list:
        if year == '2011':
            year_dict = h2_lines_dict_2011
        elif year == '2013':
            year_dict = h2_lines_dict_2013
        elif year == '2019': 
            year_dict = h2_lines_dict_2019
        for progressions in progressions_list:
            df_list = []
            if progressions in year_dict.keys():
                progression_wavelengths = h2_lines_dict_total[progressions]
                for line in progression_wavelengths:
                    line_ID = line_ID_dict[f"{line:.2f}"]
                    match = re.match(r'\((\d+)-(\d+)\)([PR])\((\d+)\)', line_ID)
                    v_exc = int(match.group(1))
                    v_ground = int(match.group(2))
                    branch = match.group(3)
                    j_ground = int(match.group(4))
                    if branch == 'R':
                        j_exc = j_ground + 1
                    elif branch == 'P':
                        j_exc = j_ground - 1
                    sub_df = pd.DataFrame([{columns_target_wavelengths[0]: float(f"{line:.2f}"),
                                 columns_target_wavelengths[1]: v_exc,
                                 columns_target_wavelengths[2]: j_exc,
                                 columns_target_wavelengths[3]: v_ground,
                                 columns_target_wavelengths[4]: j_ground}])
                    df_list.append(sub_df)
                df = pd.concat(df_list, ignore_index=True)
                file_name = r'C:\Users\zoeyates\H2_models\input_files\wavelengths\wavelengths_' +  year + '_' + progressions[1] + '_' + progressions[3] + '.sav'
                savemat(file_name, df, appendmat=False)
create_target_wavelengths()

3. target_velocity, target_flux, target_flux_err

In [15]:
def create_target_vel_file(year, progressions, file):
    if year == '2011':
        year_dict = h2_lines_dict_2011
    elif year == '2013':
        year_dict = h2_lines_dict_2013
    elif year == '2019':
        year_dict = h2_lines_dict_2019
    if progressions in year_dict.keys():
        progression_wavelengths = h2_lines_dict_total[progressions]
        vel_dict = {}
        flux_dict = {}
        flux_err_dict = {}
        load_data = np.load(file, allow_pickle=True)
        data = load_data['data'].item()
        for line in progression_wavelengths:
            line_float = float(line)
            if line_float in data[year][progressions].keys():
                wavelength_arr = data[year][progressions][line_float]['raw wl']
                vel_arr = ((wavelength_arr - line_float) / line_float) * 3e5
                vel_dict[str(line)] = vel_arr
                flux_arr = data[year][progressions][line_float]['raw flux']
                flux_dict[str(line)] = flux_arr
                flux_err_arr = data[year][progressions][line_float]['raw flux error']
                flux_err_dict[str(line)] = flux_err_arr
        if len(vel_dict) != 0:
            vel_file = (r'C:\Users\zoeyates\H2_models\input_files\vel_' + year + '_' + progressions[1] + '_' + progressions[3] + '.pkl')
            with open(vel_file, "wb") as f:
                pickle.dump(vel_dict, f)
            flux_file = (r'C:\Users\zoeyates\H2_models\input_files\flux_' + year + '_' + progressions[1] + '_' + progressions[3] + '.pkl')
            with open(flux_file, "wb") as f:
                pickle.dump(flux_dict, f)
            flux_err_file = (r'C:\Users\zoeyates\H2_models\input_files\flux_err_' + year + '_' + progressions[1] + '_' + progressions[3] + '.pkl')
            with open(flux_err_file, "wb") as f:
                pickle.dump(flux_err_dict, f)
            

In [16]:
for years in years_list:
    for progressions in progressions_list:
        create_target_vel_file(years, progressions, files_dict[progressions])